In [1]:
# ============================================================
# INSTALL LIBRARY
# ============================================================
!pip install flask pyngrok sentence-transformers

In [2]:
# ============================================================
# IMPORT LIBRARY
# ============================================================
from flask import Flask, request, jsonify
from pyngrok import ngrok
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import pandas as pd
import numpy as np
import re
import torch

In [3]:
# ============================================================
# FLASK APP
# ============================================================
app = Flask(__name__)

In [4]:
# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE :", device)

DEVICE : cpu


In [5]:
# ============================================================
# LOAD DATASET CSV
# ============================================================
df = pd.read_csv("dataset_final.csv")

print("Dataset berhasil dimuat!")
print("Jumlah data :", len(df))
print(df.head())

Dataset berhasil dimuat!
Jumlah data : 1329
   id                                               teks  label  \
0   1  Dosen memberikan materi pembelajaran melalui L...  valid   
1   2  Literasi digital membantu mahasiswa memilah in...  valid   
2   3  Mahasiswa perlu melakukan sitasi untuk menghin...  valid   
3   4  Universitas menyediakan pelatihan penggunaan p...  valid   
4   5  Mahasiswa dianjurkan menggunakan sumber jurnal...  valid   

               kategori  
0      literasi digital  
1      sumber informasi  
2  pembelajaran digital  
3      literasi digital  
4        etika akademik  


In [6]:
# ============================================================
# PREPROCESSING TEXT
# ============================================================
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = text.strip()
    return text

In [7]:
# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedder = SentenceTransformer(EMBED_MODEL)

print("Embedding Model berhasil dimuat!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model berhasil dimuat!


In [8]:
# ============================================================
# MEMBUAT EMBEDDING DATASET
# ============================================================
all_questions = df["teks"].astype(str).tolist()
all_labels = df["label"].astype(str).tolist()
all_categories = df["kategori"].astype(str).tolist()

print("Total data :", len(all_questions))

question_embeddings = embedder.encode(
    all_questions,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding selesai dibuat!")

Total data : 1329


Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Embedding selesai dibuat!


In [9]:
# ============================================================
# MENCARI DATA PALING MIRIP
# ============================================================
def retrieve_best_context(user_question):

    user_question = preprocess_text(user_question)

    user_embedding = embedder.encode(
        [user_question],
        convert_to_numpy=True
    )

    similarities = cosine_similarity(
        user_embedding,
        question_embeddings
    )[0]

    best_index = np.argmax(similarities)
    best_score = similarities[best_index]

    return {
        "text": all_questions[best_index],
        "label": all_labels[best_index],
        "kategori": all_categories[best_index],
        "score": float(best_score)
    }

In [10]:
# ============================================================
# HOME ROUTE
# ============================================================
@app.route("/")
def home():
    return """
    <html>
    <head>
        <title>Chatbot Klasifikasi Informasi</title>
        <style>
            body {
                font-family: Arial;
                background-color: #f4f6f8;
                padding: 30px;
            }
            .container {
                max-width: 700px;
                margin: auto;
                background: white;
                padding: 25px;
                border-radius: 10px;
                box-shadow: 0 0 10px rgba(0,0,0,0.1);
            }
            input {
                width: 80%;
                padding: 10px;
            }
            button {
                padding: 10px 15px;
                background: #2b7cff;
                color: white;
                border: none;
                border-radius: 5px;
            }
            #result {
                margin-top: 20px;
                background: #eef3ff;
                padding: 15px;
                border-radius: 8px;
            }
        </style>
    </head>

    <body>
        <div class="container">
            <h2>Chatbot Klasifikasi Informasi Valid dan Tidak Valid</h2>
            <p>Masukkan informasi seputar literasi digital dan pendidikan universitas.</p>

            <input type="text" id="message" placeholder="Tulis pertanyaan atau informasi...">
            <button onclick="sendMessage()">Kirim</button>

            <div id="result"></div>
        </div>

        <script>
            function sendMessage() {
                let message = document.getElementById("message").value;

                fetch("/chat", {
                    method: "POST",
                    headers: {
                        "Content-Type": "application/json"
                    },
                    body: JSON.stringify({
                        message: message
                    })
                })
                .then(response => response.json())
                .then(data => {
                    document.getElementById("result").innerHTML =
                        "<b>Hasil:</b> " + data.answer + "<br>" +
                        "<b>Kategori:</b> " + data.kategori + "<br>" +
                        "<b>Data Mirip:</b> " + data.dataset_text + "<br>" +
                        "<b>Similarity:</b> " + data.similarity;
                });
            }
        </script>
    </body>
    </html>
    """

In [11]:
# ============================================================
# CHAT API
# ============================================================
@app.route("/chat", methods=["POST"])
def chat():

    data = request.get_json()
    user_question = data["message"]

    retrieved = retrieve_best_context(user_question)
    similarity_score = retrieved["score"]

    if similarity_score < 0.40:
        return jsonify({
            "answer": "Maaf, informasi belum ditemukan dalam dataset.",
            "kategori": "-",
            "dataset_text": "-",
            "similarity": round(similarity_score, 4)
        })

    label = retrieved["label"]
    kategori = retrieved["kategori"]
    teks_dataset = retrieved["text"]

    if label == "valid":
        hasil = "Informasi VALID"
    else:
        hasil = "Informasi TIDAK VALID"

    return jsonify({
        "answer": hasil,
        "kategori": kategori,
        "dataset_text": teks_dataset,
        "similarity": round(similarity_score, 4)
    })

In [12]:
# ============================================================
# START NGROK
# ============================================================
ngrok.set_auth_token("3DZGqsANRfm9DBoqp5jViiylCs0_3ZPQq3oP6BG4YQzv1f3SS")

public_url = ngrok.connect(5000)

print("Link chatbot:", public_url)

app.run(
    host="0.0.0.0",
    port=5000
)

Link chatbot: NgrokTunnel: "https://parakeet-scone-smasher.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [14/May/2026 15:32:41] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/May/2026 15:32:45] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [14/May/2026 15:33:08] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/May/2026 15:33:09] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/May/2026 15:33:13] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/May/2026 15:33:26] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/May/2026 15:33:39] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [14/May/2026 15:34:32] "POST /chat HTTP/1.1" 200 -
